In [ ]:
## imports module
import anndata as ad
import scanpy as sc
import gc, os
import sys
from rich import print
import numpy as np
import psutil
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sea
import harmonypy as hm
import os
import psutil, gc
import matplotlib as mpl

from rich import print

In [ ]:
## general setting
sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80)
pd.set_option('display.max_columns', None)

# check how many memory used
print(psutil.virtual_memory())
# Force garbage collection
gc.collect()  
sc.settings.set_figure_params(dpi=200, facecolor="white")
PATH = "/home/kibr/Work/Vascular_atlas"

## set path and parameters
os.chdir(PATH)
sc.set_figure_params(figsize=(6, 6), frameon=False)

In [ ]:
## --- Load data --- ##
adata = sc.read_h5ad(PATH + "/Results/Revision_2/clean_object_revision_04242026.h5ad")

## subset the adata to only include cell types of interest
adata = adata[adata.obs['Cell_class'].isin(['Microglia_Macrophage_T'])].copy()

print(adata)
print(adata.X[:10,:10])

In [ ]:
## save backup
adata.raw = adata.copy()

sc.pp.filter_genes(adata, min_cells=20) 

print(adata)
print(adata.X[:10,:10])

### Performing another round of integration using Harmony

In [ ]:
## normalization
sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X.copy()

print(adata.X[:8,:8])

In [ ]:
## to reach the best results, using the below setting
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    subset=False,
    layer="log1p",
    flavor="seurat",
    batch_key="individualID",
)
## only using the highly variable genes for integration task
adata = adata[:,adata.var["highly_variable"]].copy()
print(adata)

In [ ]:
### pca and integration
print("Running PCA")
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50, svd_solver="arpack")
print(psutil.virtual_memory())

In [ ]:
## Harmony integration 
print(adata)
print("Applying Harmony integration")
sc.external.pp.harmony_integrate(adata, key='individualID', max_iter_harmony=50,basis='X_pca')
rep = "X_pca_harmony"

print(psutil.virtual_memory())

In [ ]:
print("Computing neighbors")
rep = "X_pca_harmony"
sc.pp.neighbors(adata, use_rep=rep,metric="cosine")

print("Computing leiden")
sc.tl.leiden(adata,resolution=1.0,key_added='leiden_harmony')

In [ ]:
# Check integration figures for remaining batch(individual effect)
sc.settings.figdir = f"{PATH}/Results/Revision_2/Figures"
os.makedirs(sc.settings.figdir, exist_ok=True)

sc.tl.umap(
    adata,
    # reuse existing neighbors graph
    random_state=42,
    key_added="umap_harmony"
)

sc.pl.embedding(
    adata,
    basis="umap_harmony", 
    color=["leiden_harmony", "individualID",'brain_region'],
    # color=["leiden_harmony", "individualID",'brain_region','Cell_type'],
    frameon=False,
    ncols=3,
    size=20,
    legend_loc="on data",
    save=f"_Microglia_umap_harmony.svg"
)

In [ ]:
### Given some known cell type markers, annotate the clusters
adata = adata.raw.to_adata()
adata.raw = adata.copy()
print(adata.X[:10,:10])

sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)
# sc.pp.scale(adata, max_value=10)

markers = ["F13A1","CD163","CD8A","BCL11B","CD247","PAX5","MS4A1","P2RY12","ADGRE5","THEMIS","PDCD1","CXCR5"]
# markers = {'1.1':"KCNT2",'1.2':"LAMA2",'1.3':'EPS8','2.1':"SLC4A4",'2.2':"SLIT2",'2.3':'SLC26A2','3.1':"KCNIP1",'3.2':"RNF220",'3.3':"LAMC3",'4.1':"CARMN",'4.2':"RGS6",'4.3':'PRKG1','5.1':'TRPM3','5.2':'SLC13A3'}
sc.pl.dotplot(adata, markers, groupby='leiden_harmony',use_raw=False)#, save=f"_Microglia_markers_dotplot.svg")

In [ ]:
## Checking expression of some known markers in the dotplot to annotate the clusters
sc.pl.embedding(
    adata,
    basis="umap_harmony",
    color=["F13A1","CD163","CD8A","BCL11B","CD247","PAX5","MS4A1","P2RY12","ADGRE5"],
    frameon=False,
    use_raw=False,
    ncols=5,
    size=20,
    legend_loc="on data",
    # save=f"_Fibroblast_umap_markers_harmony.svg"
)

In [ ]:
### wilcox test to find the top markers for each cluster
sc.tl.rank_genes_groups(adata, groupby='leiden_harmony', method='wilcoxon', key_added='rank_genes_harmony')
### show the top 10 markers for each cluster
result = adata.uns['rank_genes_harmony']
groups = result['names'].dtype.names
top_markers = pd.DataFrame({group: result['names'][group][:10] for group in groups})
print(top_markers)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata,
    n_genes=5,
    key='rank_genes_harmony',
    groupby='leiden_harmony',
    standard_scale='var',      # 'var' (per gene) or 'obs' (per cell)
    var_group_rotation=45,     # rotate gene labels
    colorbar_title='Mean\nexpression',
    size_title='Fraction\nexpressing',
    figsize=(15, 7),
    # save='_marker_dotplot.pdf'  # optional save
)

In [ ]:
## make the results of the wilcox test into a dataframe including the p value, adjusted p-values and log fold changes
wilcox_results = []
for cluster in groups:
    for gene, pval, pval_adj, logfc in zip(result['names'][cluster], result['pvals'][cluster], result['pvals_adj'][cluster], result['logfoldchanges'][cluster]):
        wilcox_results.append({'cluster': cluster, 'gene': gene, 'pval': pval, 'pval_adj': pval_adj, 'logfc': logfc})
wilcox_df = pd.DataFrame(wilcox_results)
display(wilcox_df.head())

## showing the top 20 markers with smallest adjusted p-values in dataframe for each cluster and the log fold changes should be greater than 0.5
top20_markers_df = wilcox_df[wilcox_df['logfc'] > 0.5].groupby('cluster').apply(lambda x: x.nsmallest(20, 'pval_adj')).reset_index(drop=True)
display(top20_markers_df)

top20_markers_df.to_csv(f"{PATH}/Results/Revision_2/DEG/Microglia_top20_markers_harmony.csv", index=False)

In [ ]:
### Try to cluster the groups and run DEG again for checking
cluster2celltype = {
    "0": "Microglia",
    "1": "Microglia",
    "2": "Microglia",
    "3": "Microglia",
    "4": "Microglia",
    "5": "Microglia",
    "6": "Macrophage",
    "7": "Microglia",
    "8": "Microglia",
    "9": "Microglia",
    "10": "Microglia",
    "11": "Monocyte",
    "12": "T-cell",
}
adata.obs['Cell_type'] = adata.obs['leiden_harmony'].map(cluster2celltype)

sc.pl.embedding(
    adata,
    basis='umap_harmony',   # <- THIS picks .obsm['X_' + key]
    # color=['leiden_harmony','Cell_type'],
    color = ['brain_region','region_layer','Cell_type'],
    frameon=False,
    ncols=1,
    size=20,
    # legend_loc="on data",
    use_raw = False,
    save=f"_Microglia_umap_CT_RG.svg"
    )

In [ ]:
### Given some known cell type markers, annotate the clusters
mpl.rcParams['svg.fonttype'] = 'none'
adata = adata.raw.to_adata()
adata.raw = adata.copy()
print(adata.X[:10,:10])

sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)

markers = ["F13A1","CD163",'P2RY12','ADGRE5',"CD247"]

sc.pl.dotplot(adata, markers, groupby='Cell_type',use_raw=False, save=f"_Microglia_markers_dotplot.svg")

In [ ]:
### ---------------------------------------------------------------------
### Saving processed data
### ---------------------------------------------------------------------
adata = adata.raw.to_adata()

print(adata)
print(adata.X[:10,:10])
## Saving h5ad
results_file = PATH+"/Results/Revision_2/Microglia_temp_processed.h5ad"
adata.write(results_file)
print("Done. Saved:", results_file)

## Builting region-level representation (Donor-balanced centroids in PC spave)

In [ ]:
### Reloading the data 
mpl.rcParams['svg.fonttype'] = 'none'
adata = sc.read_h5ad(PATH+"/Results/Revision_2/Microglia_temp_processed.h5ad")
print(adata)
print(adata.X[:10,:10])

sc.pl.embedding(
    adata,
    basis='umap_harmony',   # <- THIS picks .obsm['X_' + key]
    color=['leiden_harmony','individualID','Cell_type'],
    frameon=False,
    ncols=4,
    size=20,
    legend_loc="on data",
    use_raw = False,
    save=f"_Microglia_CT.svg"
    )

## Draw a stacked barplot showing cell type proportion across brain regions

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
region_meta = (
    pd.read_csv("./Data/region_update.csv")
    .assign(
        Region_layer_1=lambda df: df["Region_layer_1"].str.strip().str.title(),
        Region_layer_4=lambda df: df["Region_layer_2"].str.strip()
    )
    [["Region_layer_1", "Region_layer_2"]]
)
# region_meta

In [ ]:
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
# ── 1. Get metadata ───────────────────────────────────────────────────────────
meta = adata.obs.copy()
meta["region_name"] = meta["region_name"].str.strip().str.title()
meta["brain_region"] = meta["regionID"].astype(str) + "_" + meta["region_name"]
# meta.loc[meta["region_name"] == "Mid Temporal Gyrus", "region_name"] = "Middle Temporal Gyrus"

# ── 2. Count table (cell_type instead of cell_class) ─────────────────────────
counts = (
    meta.groupby(["Cell_type", "brain_region"])
    .size()
    .reset_index(name="freq")
)
counts["brain_region"] = counts["brain_region"].str.replace("Mid Temporal Gyrus", "Middle Temporal Gyrus")

# ── 3. Parse order and region name ───────────────────────────────────────────
counts[["order", "brain_region_plot"]] = counts["brain_region"].str.split("_", n=1, expand=True)
counts["order"] = pd.to_numeric(counts["order"], errors="coerce")

# ── 4. Join region_layer_4 ────────────────────────────────────────────────────
counts = counts.merge(region_meta, left_on="brain_region_plot", right_on="Region_layer_1", how="left")

# ── 5. Sort by layer4 group, then numeric region order ───────────────────────
layer2_order = [
  "Cortex",
  "Limbic",
  "White Matter Tracts",
  "Brainstem",
  "Cerebellum",
  "Watershed",
  "Major Vessel",
  "Olfactory",
  "Barrier",
#   "Subcortical"
]
counts["layer2_order_idx"] = counts["Region_layer_2"].map({v: i for i, v in enumerate(layer2_order)})
counts = counts.sort_values(["layer2_order_idx", "order"]).reset_index(drop=True)
region_levels = list(dict.fromkeys(counts["brain_region_plot"]))

# ── 6. Region color map for tick labels ───────────────────────────────────────
region_color_map = {
    "Cortex": "#009E73",
    "Brainstem": "#D55E00",
    "Cerebellum": "#046299",
    "Limbic": "#03B8D8",
    "Watershed": "#E69F00",
    "White Matter Tracts": "#CC79A7",
    "Olfactory": "#999999",
    "Barrier": "#9C0AE0",
    # "Subcortical": "#915330",
    "Major Vessel": "#FF0000"
}

rl4_lookup = (
    counts[["brain_region_plot", "Region_layer_2"]]
    .drop_duplicates()
    .set_index("brain_region_plot")["Region_layer_2"]
    .to_dict()
)
tick_colors = [region_color_map.get(str(rl4_lookup.get(r, "")), "black") for r in region_levels]

# ── 7. Pivot and normalize to proportions ─────────────────────────────────────
bar_data = (
    counts.pivot_table(index="brain_region_plot", columns="Cell_type", values="freq", aggfunc="sum")
    .fillna(0)
    .reindex(region_levels)
)
bar_data = bar_data.div(bar_data.sum(axis=1), axis=0)  # row-wise proportion
cell_types = bar_data.columns.tolist()

# ── 8. Plot ───────────────────────────────────────────────────────────────────
n_regions = len(region_levels)
x_pos     = np.arange(n_regions)
bottoms   = np.zeros(n_regions)

fig, ax = plt.subplots(figsize=(14, 6))

for cell in cell_types:
    vals = bar_data[cell].values
    ax.bar(x_pos, vals, bottom=bottoms, label=cell, width=0.8)  # matplotlib auto-assigns colors
    bottoms += vals

# X-axis labels with region colors
ax.set_xticks(x_pos)
ax.set_xticklabels(region_levels, rotation=90, ha="right", va="top", fontsize=15)
for tick, color in zip(ax.get_xticklabels(), tick_colors):
    tick.set_color(color)

ax.set_xlim(-0.5, n_regions - 0.5)
ax.set_ylabel("Cell count", fontsize=14)
ax.set_xlabel("")
ax.grid(axis="x", visible=False)
ax.grid(axis="y", linestyle="--", alpha=0.4)

ax.legend(loc="upper right",bbox_to_anchor=(1.2, 1), ncol=1, fontsize=15, frameon=True,
          title="Cell type", title_fontsize=8)

plt.tight_layout()
plt.savefig("./Results/Revision_2/Figures/Sup_Fig_x_Microglia_cell_type_proportion.pdf", bbox_inches="tight", dpi=300)
plt.show()

## Calculate the region-pairwise distance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import pdist, squareform

In [ ]:
# ── 1. Extract metadata ────────────────────────────────────────────────────────
meta = adata.obs.copy()

# ── 2. Count table: clusters × brain regions ──────────────────────────────────
counts = (meta.groupby(["leiden_harmony", "region_name"])
              .size()
              .reset_index(name="freq"))
counts.columns = ["cluster", "brain_region", "freq"]

counts["brain_region"] = pd.Categorical(counts["brain_region"],
                                         categories=counts["brain_region"].unique())
counts["cluster"] = counts["cluster"].astype("category")

# ── 3. Wide matrix → proportions ──────────────────────────────────────────────
comp_matrix = counts.pivot_table(index="brain_region", columns="cluster",
                                  values="freq", fill_value=0)
comp_matrix = comp_matrix.div(comp_matrix.sum(axis=1), axis=0)  # row proportions

# ── 4. Hellinger distance ─────────────────────────────────────────────────────
hellinger = np.sqrt(comp_matrix.values)
dist_mat = squareform(pdist(hellinger, metric="euclidean"))
dist_matrix = pd.DataFrame(dist_mat,
                            index=comp_matrix.index,
                            columns=comp_matrix.index)


In [ ]:
# ── 5. Region color mapping ───────────────────────────────────────────────────
mpl.rcParams['svg.fonttype'] = 'none'

layer_colors = {
    "Cortex": "#009E73",
    "Brainstem": "#D55E00",
    "Cerebellum": "#046299",
    "Limbic": "#03B8D8",
    "Watershed": "#E69F00",
    "White Matter Tracts": "#CC79A7",
    "Olfactory": "#999999",
    "Barrier": "#9C0AE0",
    # "Subcortical": "#915330",
    "Major Vessel": "#FF0000",
}

df_colors = meta[["region_name", "region_layer"]].drop_duplicates()
df_colors["region_color"] = df_colors["region_layer"].map(layer_colors)
region_color = df_colors.set_index("region_name")["region_color"].to_dict()

# ── 6. Row color bar (left annotation) ───────────────────────────────────────
row_colors = pd.Series([region_color.get(r, "#ffffff")
                         for r in dist_matrix.index],
                        index=dist_matrix.index,
                        name="Region")

# ── 7. Clustermap (equivalent to ComplexHeatmap) ─────────────────────────────
np.random.seed(42)

cmap = plt.get_cmap("magma_r")   # rev(magma(100))

g = sns.clustermap(
    dist_matrix,
    # metric="precomputed",       # distance matrix is already computed
    method="average",
    cmap=cmap,
    row_colors=row_colors,
    col_colors=row_colors,
    figsize=(14, 14),
    xticklabels=True,
    yticklabels=True,
    cbar_kws={"label": "Distance"},
)

g.fig.suptitle("Pairwise Distances Between Regions", y=1.01)
g.ax_heatmap.grid(False)
# ── 8. Save ───────────────────────────────────────────────────────────────────
g.savefig("./Results/Revision_2/Figures/Microglia_composition_distance_heatmap.pdf",
          bbox_inches="tight")
plt.show()